In [2]:
pip install anthropic python-dotenv -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
import time
import pandas as pd
from pathlib import Path
from anthropic import Anthropic
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv("../.env")

# Verify the key is loaded (don't print it!)
assert os.getenv("ANTHROPIC_API_KEY"), "API key not found in .env file"
print("✓ API key loaded successfully")
print(f"Key starts with: {os.getenv('ANTHROPIC_API_KEY')[:15]}...")

# Initialize the client
client = Anthropic()
MODEL = "claude-sonnet-4-5"
print(f"✓ Client initialized with model: {MODEL}")

✓ API key loaded successfully
Key starts with: sk-ant-api03-1a...
✓ Client initialized with model: claude-sonnet-4-5


In [4]:
OUTPUTS_DIR = Path("../Outputs")
master = pd.read_csv(OUTPUTS_DIR / "routed_master_905.csv")

# Posts that need LLM (yellow or red in either emotion or toxicity)
needs_llm = master[
    (master["emotion_bucket"].isin(["yellow", "red"])) |
    (master["toxicity_bucket"].isin(["yellow", "red"]))
].copy()

print(f"Total posts in master: {len(master)}")
print(f"Posts needing LLM scoring: {len(needs_llm)}")
print()
print("Breakdown:")
print(f"  Emotion yellow: {(master['emotion_bucket'] == 'yellow').sum()}")
print(f"  Emotion red:    {(master['emotion_bucket'] == 'red').sum()}")
print(f"  Toxicity yellow: {(master['toxicity_bucket'] == 'yellow').sum()}")
print(f"  Toxicity red:    {(master['toxicity_bucket'] == 'red').sum()}")

Total posts in master: 905
Posts needing LLM scoring: 697

Breakdown:
  Emotion yellow: 696
  Emotion red:    0
  Toxicity yellow: 83
  Toxicity red:    1


In [5]:
EMOTION_OBJECTIVE = """You are analyzing a social media post for academic research.

Read this post and analyze its linguistic content. What emotions does the post EXPRESS? Score each of the 6 emotions from 0.0 to 1.0 based on the post's actual emotional tone, not just keywords. A post describing or reporting on an emotion is different from a post expressing it.

The 6 emotions:
- anger: hostility, rage, frustration
- disgust: revulsion, contempt, aversion
- fear: anxiety, dread, threat
- joy: happiness, delight, excitement
- sadness: grief, sorrow, melancholy
- surprise: shock, astonishment

Scoring: 0.0 = absent, 0.5 = moderately present, 1.0 = dominant. Use any decimal in between. Multiple emotions can co-exist.

Also rate your own confidence: how certain are you that your scores accurately capture this post's emotional content? Consider factors like text length, clarity, ambiguity. Be honest — 1.0 means very certain, 0.5 means moderately uncertain, 0.0 means basically guessing. Do not default to high confidence.

Post:
\"\"\"
{text}
\"\"\"

Respond with ONLY valid JSON in this exact format:
{{
  "anger": 0.0,
  "disgust": 0.0,
  "fear": 0.0,
  "joy": 0.0,
  "sadness": 0.0,
  "surprise": 0.0,
  "confidence": 0.0,
  "reasoning": "One sentence explaining the dominant emotion and your reasoning."
}}"""


EMOTION_YOUTH = """Imagine you ARE a 13-17 year old scrolling social media. You see this post on your feed. How would you FEEL?

Score each emotion from 0.0 to 1.0 based on your gut reaction as a young viewer.

The 6 emotions (how a teen would feel them):
- anger: irritated, mad, hostile
- disgust: grossed out, contemptuous, "ew"
- fear: anxious, worried, unsettled
- joy: happy, hyped, smiling, excited
- sadness: down, empathetically sad, blue
- surprise: shocked, "wait what", unexpected

Scoring: 0.0 = wouldn't feel it, 0.5 = moderate feeling, 1.0 = strong feeling. Use any decimal in between. Multiple feelings can occur at once.

Also rate your own confidence in your scores from 0.0 to 1.0. Be honest — do not default to high confidence.

Post you see:
\"\"\"
{text}
\"\"\"

Respond with ONLY valid JSON in this exact format:
{{
  "anger": 0.0,
  "disgust": 0.0,
  "fear": 0.0,
  "joy": 0.0,
  "sadness": 0.0,
  "surprise": 0.0,
  "confidence": 0.0,
  "reasoning": "One sentence explaining your gut reaction as a 13-17 year old viewer."
}}"""


TOXICITY_OBJECTIVE = """You are analyzing a social media post for academic research.

Read this post and analyze its linguistic content. What toxicity does the post EXPRESS? Score each dimension from 0.0 to 1.0 based on actual harm, not just the presence of strong words. Casual profanity in entertainment is NOT the same as harassment.

Toxicity dimensions:
- overall_toxic: harmful, abusive, or threatening content
- insult: directly insulting a person or group
- threat: threatening violence or harm
- identity_attack: attacks based on race, gender, religion, etc.
- profanity: contains swear words (regardless of intent)
- harassment: bullying, targeted hostility

Scoring: 0.0 = absent, 0.5 = moderately present, 1.0 = strongly present. Use any decimal in between. Multiple dimensions can co-exist.

A gym vlog with "fucking weights" scores high profanity but low overall_toxic. A targeted personal attack scores high on overall_toxic AND insult.

Also rate your own confidence in your scores from 0.0 to 1.0. Be honest — do not default to high confidence.

Post:
\"\"\"
{text}
\"\"\"

Respond with ONLY valid JSON in this exact format:
{{
  "overall_toxic": 0.0,
  "insult": 0.0,
  "threat": 0.0,
  "identity_attack": 0.0,
  "profanity": 0.0,
  "harassment": 0.0,
  "confidence": 0.0,
  "reasoning": "One sentence explaining the toxicity assessment."
}}"""


TOXICITY_YOUTH = """Imagine you ARE a 13-17 year old scrolling social media. You see this post on your feed. How would you FEEL about it?

Score each dimension from 0.0 to 1.0 based on whether YOU would feel uncomfortable, threatened, or that the post is being mean.

Dimensions:
- feels_uncivil: would feel rude or hostile to you
- feels_insulting: would feel like it's putting someone down
- feels_threatening: would make you uncomfortable or scared
- feels_attacking: would feel like targeting a specific group
- feels_profane: contains swearing that bothers you
- feels_bullying: would feel like harassment

Scoring: 0.0 = wouldn't perceive it this way, 0.5 = moderately, 1.0 = strongly. Use any decimal in between.

Note: Teens are usually comfortable with profanity in entertainment and don't perceive it as toxic. Score based on actual teen perception.

Also rate your own confidence in your scores from 0.0 to 1.0. Be honest — do not default to high confidence.

Post you see:
\"\"\"
{text}
\"\"\"

Respond with ONLY valid JSON in this exact format:
{{
  "feels_uncivil": 0.0,
  "feels_insulting": 0.0,
  "feels_threatening": 0.0,
  "feels_attacking": 0.0,
  "feels_profane": 0.0,
  "feels_bullying": 0.0,
  "confidence": 0.0,
  "reasoning": "One sentence explaining your gut reaction as a 13-17 year old viewer."
}}"""

print("✓ All 4 prompts defined (emotion objective + youth, toxicity objective + youth)")

✓ All 4 prompts defined (emotion objective + youth, toxicity objective + youth)


In [6]:
# Test on one post to make sure everything works
test_text = needs_llm.iloc[0]["text_for_analysis"]
print(f"Test post: {test_text[:200]}\n")

response = client.messages.create(
    model=MODEL,
    max_tokens=400,
    temperature=1.0,
    messages=[{
        "role": "user",
        "content": EMOTION_OBJECTIVE.format(text=test_text[:2000])
    }]
)

print("=== Raw response ===")
print(response.content[0].text)
print(f"\n=== Token usage ===")
print(f"Input tokens: {response.usage.input_tokens}")
print(f"Output tokens: {response.usage.output_tokens}")

# Estimate cost
input_cost = response.usage.input_tokens * 3 / 1_000_000
output_cost = response.usage.output_tokens * 15 / 1_000_000
print(f"\n=== Cost for this call ===")
print(f"${input_cost + output_cost:.5f}")

Test post: Chinese authorities say their investigation of a high-profile scandal at one of the countrys leading state-run museums has revealed systemic mismanagement and alleged corruption over decades. #nanjin

=== Raw response ===
```json
{
  "anger": 0.0,
  "disgust": 0.0,
  "fear": 0.0,
  "joy": 0.0,
  "sadness": 0.0,
  "surprise": 0.0,
  "confidence": 0.85,
  "reasoning": "This is a straightforward news report describing a corruption scandal with neutral, journalistic language that reports on events without expressing emotional tone; it informs rather than reacts emotionally."
}
```

=== Token usage ===
Input tokens: 759
Output tokens: 125

=== Cost for this call ===
$0.00415


In [6]:
def score_post(text, prompt_template, max_retries=3):
    """Run one post through one prompt. Returns parsed JSON or None."""
    text = str(text)[:2000]
    
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=400,
                temperature=1.0,
                messages=[{
                    "role": "user",
                    "content": prompt_template.format(text=text)
                }]
            )
            raw = response.content[0].text.strip()
            
            # Strip code fences if present
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
                raw = raw.strip()
            
            parsed = json.loads(raw)
            
            # Ensure required fields exist (LLM sometimes forgets)
            if "confidence" not in parsed:
                parsed["confidence"] = None
            if "reasoning" not in parsed:
                parsed["reasoning"] = ""
            
            parsed["_input_tokens"] = response.usage.input_tokens
            parsed["_output_tokens"] = response.usage.output_tokens
            return parsed
        except json.JSONDecodeError as e:
            print(f"  JSON parse error on attempt {attempt+1}: {e}")
            time.sleep(1)
        except Exception as e:
            print(f"  API error on attempt {attempt+1}: {e}")
            time.sleep(2 ** attempt)
    
    return None

In [8]:
# Pilot test: run 10 yellow posts × 4 prompts = 40 calls
pilot_posts = needs_llm.head(10).copy()

PILOT_TASKS = [
    ("emotion_objective", EMOTION_OBJECTIVE),
    ("emotion_youth", EMOTION_YOUTH),
    ("toxicity_objective", TOXICITY_OBJECTIVE),
    ("toxicity_youth", TOXICITY_YOUTH),
]

pilot_results = []
total_cost = 0.0

for idx, row in pilot_posts.iterrows():
    print(f"\nPost {idx}: {str(row['text_for_analysis'])[:80]}...")
    for task_name, prompt in PILOT_TASKS:
        result = score_post(row["text_for_analysis"], prompt)
        if result is None:
            print(f"  {task_name}: FAILED")
            continue
        
        cost = result["_input_tokens"] * 3 / 1_000_000 + result["_output_tokens"] * 15 / 1_000_000
        total_cost += cost
        
        result["_post_index"] = idx
        result["_task"] = task_name
        result["_text_source"] = row["text_source"]
        pilot_results.append(result)
        print(f"  {task_name}: confidence={result.get('confidence', 'N/A')}, reasoning={result.get('reasoning', '')[:80]}")

print(f"\n=== Pilot complete ===")
print(f"Total API calls: {len(pilot_results)}")
print(f"Total cost: ${total_cost:.4f}")
print(f"Average cost per call: ${total_cost/len(pilot_results):.5f}")
print(f"\nEstimated cost for full run ({len(needs_llm)} posts × 4 prompts = {len(needs_llm)*4} calls):")
print(f"  ${total_cost/len(pilot_results) * len(needs_llm) * 4:.2f}")


Post 0: Chinese authorities say their investigation of a high-profile scandal at one of ...
  emotion_objective: confidence=0.85, reasoning=This is a neutral journalistic news report describing a corruption scandal witho
  emotion_youth: confidence=N/A, reasoning=As a teen, I'd feel kind of grossed out that people in charge stole important cu
  toxicity_objective: confidence=0.95, reasoning=This is a straightforward news report about a corruption scandal at a Chinese mu
  toxicity_youth: confidence=0.9, reasoning=This is just a straightforward news article about a museum scandal in China that

Post 1: You might be feeling overwhelmed with everything thats going on lately
.
Refra...
  emotion_objective: confidence=0.7, reasoning=The post expresses moderate joy through its encouraging, uplifting tone with sup
  emotion_youth: confidence=0.7, reasoning=This feels like generic "bro" motivational content that's trying too hard and co
  toxicity_objective: confidence=0.95, reasoning=This 

In [10]:
# Full run with checkpoint saving
def run_task(df, task_name, prompt_template, output_path, save_every=20):
    """Run all posts through one prompt with incremental saving."""
    
    # Resume from checkpoint if exists
    done_indices = set()
    if Path(output_path).exists():
        existing = pd.read_csv(output_path)
        done_indices = set(existing["_post_index"].astype(int))
        print(f"  Resuming: {len(done_indices)} already done")
    
    new_results = []
    batch_count = 0
    
    for idx, row in df.iterrows():
        if idx in done_indices:
            continue
        
        result = score_post(row["text_for_analysis"], prompt_template)
        if result is None:
            result = {"_failed": True}
        
        result["_post_index"] = idx
        result["_student_id"] = row["student_id"]
        result["_text_source"] = row["text_source"]
        new_results.append(result)
        batch_count += 1
        
        # Save checkpoint
        if batch_count % save_every == 0:
            if Path(output_path).exists():
                existing = pd.read_csv(output_path)
                combined = pd.concat([existing, pd.DataFrame(new_results)], ignore_index=True)
            else:
                combined = pd.DataFrame(new_results)
            combined.to_csv(output_path, index=False)
            new_results = []
            print(f"  Checkpoint saved: {len(combined)} total")
    
    # Final save
    if new_results:
        if Path(output_path).exists():
            existing = pd.read_csv(output_path)
            combined = pd.concat([existing, pd.DataFrame(new_results)], ignore_index=True)
        else:
            combined = pd.DataFrame(new_results)
        combined.to_csv(output_path, index=False)
    
    print(f"  DONE: {output_path}")


# Define all 4 tasks
TASKS = [
    ("emotion_objective", EMOTION_OBJECTIVE, OUTPUTS_DIR / "llm_emotion_objective_905.csv"),
    ("emotion_youth", EMOTION_YOUTH, OUTPUTS_DIR / "llm_emotion_youth_905.csv"),
    ("toxicity_objective", TOXICITY_OBJECTIVE, OUTPUTS_DIR / "llm_toxicity_objective_905.csv"),
    ("toxicity_youth", TOXICITY_YOUTH, OUTPUTS_DIR / "llm_toxicity_youth_905.csv"),
]

# Filter to subset for each task
for task_name, prompt, output_path in TASKS:
    print(f"\n=== Running {task_name} ===")
    
    # Emotion tasks → emotion yellow/red posts
    # Toxicity tasks → toxicity yellow/red posts
    if "emotion" in task_name:
        subset = master[master["emotion_bucket"].isin(["yellow", "red"])].copy()
    else:
        subset = master[master["toxicity_bucket"].isin(["yellow", "red"])].copy()
    
    print(f"  Posts to process: {len(subset)}")
    run_task(subset, task_name, prompt, output_path, save_every=20)


=== Running emotion_objective ===
  Posts to process: 696
  Checkpoint saved: 20 total
  Checkpoint saved: 40 total
  Checkpoint saved: 60 total
  Checkpoint saved: 80 total
  Checkpoint saved: 100 total
  Checkpoint saved: 120 total
  Checkpoint saved: 140 total
  Checkpoint saved: 160 total
  Checkpoint saved: 180 total
  Checkpoint saved: 200 total
  Checkpoint saved: 220 total
  Checkpoint saved: 240 total
  Checkpoint saved: 260 total
  Checkpoint saved: 280 total
  Checkpoint saved: 300 total
  Checkpoint saved: 320 total
  Checkpoint saved: 340 total
  Checkpoint saved: 360 total
  Checkpoint saved: 380 total
  Checkpoint saved: 400 total
  Checkpoint saved: 420 total
  Checkpoint saved: 440 total
  Checkpoint saved: 460 total
  Checkpoint saved: 480 total
  Checkpoint saved: 500 total
  Checkpoint saved: 520 total
  Checkpoint saved: 540 total
  Checkpoint saved: 560 total
  Checkpoint saved: 580 total
  Checkpoint saved: 600 total
  Checkpoint saved: 620 total
  Checkpoint sa

In [8]:
OUTPUTS_DIR = Path("../Outputs")

TASKS = [
    ("emotion_objective", OUTPUTS_DIR / "llm_emotion_objective_905.csv"),
    ("emotion_youth", OUTPUTS_DIR / "llm_emotion_youth_905.csv"),
    ("toxicity_objective", OUTPUTS_DIR / "llm_toxicity_objective_905.csv"),
    ("toxicity_youth", OUTPUTS_DIR / "llm_toxicity_youth_905.csv"),
]

print("=== LLM Scoring Summary ===\n")
for task_name, output_path in TASKS:
    if output_path.exists():
        df = pd.read_csv(output_path)
        failed = df["_failed"].sum() if "_failed" in df.columns else 0
        print(f"{task_name}:")
        print(f"  Total scored: {len(df)}")
        print(f"  Failed: {failed}")
        if "confidence" in df.columns:
            avg_conf = df["confidence"].mean()
            high_conf = (df["confidence"] >= 0.8).sum()
            low_conf = (df["confidence"] < 0.2).sum()
            null_conf = df["confidence"].isna().sum()
            print(f"  Avg confidence: {avg_conf:.2f}")
            print(f"  Confidence ≥ 0.8: {high_conf}")
            print(f"  Confidence < 0.2: {low_conf}")
            print(f"  Confidence missing: {null_conf}")
        print()
    else:
        print(f"{task_name}: FILE NOT FOUND")
        print()

=== LLM Scoring Summary ===

emotion_objective:
  Total scored: 696
  Failed: 0
  Avg confidence: 0.56
  Confidence ≥ 0.8: 101
  Confidence < 0.2: 23
  Confidence missing: 6

emotion_youth:
  Total scored: 696
  Failed: 0
  Avg confidence: 0.63
  Confidence ≥ 0.8: 47
  Confidence < 0.2: 0
  Confidence missing: 27

toxicity_objective:
  Total scored: 84
  Failed: 0
  Avg confidence: 0.90
  Confidence ≥ 0.8: 73
  Confidence < 0.2: 0
  Confidence missing: 0

toxicity_youth:
  Total scored: 84
  Failed: 0
  Avg confidence: 0.85
  Confidence ≥ 0.8: 64
  Confidence < 0.2: 0
  Confidence missing: 0



# LLM Scoring Summary — 905 Dataset

## What we did

Sent posts that landed in the Yellow or Red routing buckets (from the transformer routing pipeline) to Claude Sonnet 4.5 for independent scoring. Each post was scored twice — once with an objective analytical prompt, and once simulating a 13-17 year old viewer's perspective. This was done for both emotion and toxicity.

## Posts processed

- **Emotion (yellow bucket):** 696 posts × 2 prompts = 1,392 calls
- **Toxicity (yellow + red bucket):** 84 posts × 2 prompts = 168 calls
- **Total API calls:** 1,560

## Cost

- Avg cost per call: $0.0034
- Total cost: ~$5.30
- Well under the $20 budget

## Files generated

In `Outputs/`:

| File | Rows | What's in it |
|---|---|---|
| `llm_emotion_objective_905.csv` | 696 | Per-post scores for 6 emotions (anger, disgust, fear, joy, sadness, surprise) + confidence + reasoning |
| `llm_emotion_youth_905.csv` | 696 | Same but from a teen perspective |
| `llm_toxicity_objective_905.csv` | 84 | Per-post scores for 6 toxicity dimensions (overall_toxic, insult, threat, identity_attack, profanity, harassment) + confidence + reasoning |
| `llm_toxicity_youth_905.csv` | 84 | Same but from a teen perspective |

Each row has `_post_index` linking back to `routed_master_905.csv`. The post text itself is in the master, not duplicated in the LLM files.

## Confidence distribution

| Task | Avg confidence | ≥ 0.8 | < 0.2 | Missing |
|---|---|---|---|---|
| emotion_objective | 0.56 | 101 | 23 | 6 |
| emotion_youth | 0.63 | 47 | 0 | 27 |
| toxicity_objective | 0.90 | 73 | 0 | 0 |
| toxicity_youth | 0.85 | 64 | 0 | 0 |

## Honest findings

**Toxicity LLM is highly confident.** 87% of toxicity posts came back with confidence ≥ 0.8. The LLM clearly understands what's toxic on social media and can distinguish casual profanity (gym videos, music lyrics) from actual harm.

**Emotion LLM is much less confident.** Only 15% of emotion posts came back at ≥ 0.8 confidence on objective scoring, 7% on the youth perspective. The LLM is honestly admitting that emotion classification on short, ambiguous social media text is hard.

**This is not a model failure — it's calibrated uncertainty.** The yellow bucket posts are by definition the hard cases (where transformers disagreed). The LLM correctly recognizes most of them are genuinely ambiguous. This is more useful than a model that always claims high confidence.

**23 emotion_objective posts have confidence below 0.2.** These are essentially "I really don't know" cases — the LLM couldn't form a meaningful judgment.

**27 emotion_youth posts are missing confidence values.** The LLM occasionally omitted the field in its JSON response. These need to be handled as "unknown confidence" in downstream consolidation.

## Implication for human review

Using the originally proposed thresholds (≥ 0.8 accept, 0.2–0.8 human review, < 0.2 drop):

- For emotion: ~570 posts would need human review out of 696. Not operationally feasible.
- For toxicity: ~11 posts would need human review out of 84. Manageable.

**Threshold needs revisiting in next meeting.** Possible alternatives:

1. Lower the auto-accept threshold to 0.6 (simpler but less defensible)
2. Use a three-tier system: high confidence (≥0.7), medium confidence (0.4–0.7, still accepted but tagged), low confidence (0.2–0.4, human review), drop (<0.2)
3. Random sampling of medium-confidence posts for spot-check instead of full human review

## What hasn't been done yet

- **Consolidation:** Merging LLM scores back to the master CSV to produce one final per-post output
- **Final scoring decisions:** Apply confidence gating once threshold is locked in next meeting
- **Misalignment analysis:** Comparing transformer scores vs LLM scores vs student self-reports (Q7/Q8/Q9)
- **Flow diagram:** Visual pipeline for the grant document

## Sample LLM reasoning (showing quality is real)

**Post:** "Chinese authorities say their investigation of a high-profile scandal..."
**LLM objective:** All emotions 0.0, confidence 0.85. *"This is a neutral journalistic news report describing a corruption scandal without expressing emotional tone; it informs rather than reacts emotionally."*
**Transformer (DistilRoBERTa):** 86% anger
**Transformer (Cardiff):** 77% anger, 82% disgust

The LLM correctly identified that the post is *reporting* on negative events, not *expressing* anger or disgust. Transformers fired on keywords like "scandal" and "corruption" without understanding the journalistic register.

This is the kind of contextual nuance LLM scoring was added to capture.

## Open questions for next meeting

1. What confidence threshold to use for auto-accept vs human review?
2. How to handle the 27 emotion_youth posts with missing confidence values?
3. How to operationalize human review at this scale (any access to annotator like Shriva)?
4. Whether to lower the routing threshold (currently 0.25) to shift more posts to Green and reduce LLM dependency, OR keep as-is for cleaner methodology.

In [1]:
import pandas as pd
from pathlib import Path

OUTPUTS_DIR = Path("../Outputs")

LLM_FILES = [
    ("emotion_objective", OUTPUTS_DIR / "llm_emotion_objective_905.csv"),
    ("emotion_youth", OUTPUTS_DIR / "llm_emotion_youth_905.csv"),
    ("toxicity_objective", OUTPUTS_DIR / "llm_toxicity_objective_905.csv"),
    ("toxicity_youth", OUTPUTS_DIR / "llm_toxicity_youth_905.csv"),
]

print("=== Counting at threshold 0.6 ===\n")
for task_name, path in LLM_FILES:
    df = pd.read_csv(path)
    total = len(df)
    
    above = (df["confidence"] >= 0.6).sum()
    below = ((df["confidence"] >= 0.2) & (df["confidence"] < 0.6)).sum()
    drop = (df["confidence"] < 0.2).sum()
    missing = df["confidence"].isna().sum()
    
    print(f"{task_name} (total: {total}):")
    print(f"  Confidence ≥ 0.6 (auto-accept): {above} ({above/total*100:.1f}%)")
    print(f"  Confidence 0.2 to 0.6 (human review): {below} ({below/total*100:.1f}%)")
    print(f"  Confidence < 0.2 (drop): {drop} ({drop/total*100:.1f}%)")
    print(f"  Missing confidence: {missing}")
    print()

=== Counting at threshold 0.6 ===

emotion_objective (total: 696):
  Confidence ≥ 0.6 (auto-accept): 420 (60.3%)
  Confidence 0.2 to 0.6 (human review): 247 (35.5%)
  Confidence < 0.2 (drop): 23 (3.3%)
  Missing confidence: 6

emotion_youth (total: 696):
  Confidence ≥ 0.6 (auto-accept): 562 (80.7%)
  Confidence 0.2 to 0.6 (human review): 107 (15.4%)
  Confidence < 0.2 (drop): 0 (0.0%)
  Missing confidence: 27

toxicity_objective (total: 84):
  Confidence ≥ 0.6 (auto-accept): 82 (97.6%)
  Confidence 0.2 to 0.6 (human review): 2 (2.4%)
  Confidence < 0.2 (drop): 0 (0.0%)
  Missing confidence: 0

toxicity_youth (total: 84):
  Confidence ≥ 0.6 (auto-accept): 83 (98.8%)
  Confidence 0.2 to 0.6 (human review): 1 (1.2%)
  Confidence < 0.2 (drop): 0 (0.0%)
  Missing confidence: 0

